# 🔵 Silver Layer – Events (Fact Table)

## 📌 Objective

Create a **fact table** by combining transactional data with dimensions.

## 📊 Data Type

* High-volume transactional data
* Streaming ingestion (Auto Loader)

## 🧠 Transformations

* Data type casting
* Derived columns (event_date)
* Success flag handling
* Partition column creation

## 🔗 Joins

* Players
* Species
* Maps
* Rods
* Baits

## ⚡ Performance

* Partitioned by `event_date`
* Optimized for query performance

## 📥 Source

* Bronze Events
* Silver Dimensions

## 📤 Target

* `game_analytics.silver.events`

## 🎯 Goal

Build a **Star Schema Fact Table**

In [0]:
from pyspark.sql import functions as sf
silver_path = "abfss://curated@storageaccgameanalytics.dfs.core.windows.net/silver"

In [0]:
events = spark.read.table('game_analytics.bronze.events')
players = spark.read.table("game_analytics.silver.players")
species = spark.read.table("game_analytics.silver.species")
maps = spark.read.table("game_analytics.silver.maps")
rods = spark.read.table("game_analytics.silver.rods")
baits = spark.read.table("game_analytics.silver.baits")

In [0]:
events_silver = (
    events
    .select(
        "event_id",
        "player_id",
        "species_id",
        "map_id",
        "rod_id",
        "bait_id",
        "success_flag",
        "event_time",
        "weight",
        "price",
        "date",
        "ingestion_time"
    )
    
    .withColumn("event_timestamp", sf.col("event_time").cast("timestamp"))
    .withColumn("weight", sf.col("weight").cast("double"))
    .withColumn("price", sf.col("price").cast("int"))
    .withColumn("success_flag", sf.col("success_flag").cast("boolean"))
    
    .withColumn("event_date", sf.to_date("date"))
    
    .withColumn(
        "catch_status",
        sf.when(sf.col("success_flag") == True, "SUCCESS")
         .otherwise("FAILED")
    )
    
    .drop("event_time", "date")
)

In [0]:
events_silver = (
    events_silver.alias("e")
    .join(players.select("player_id", "full_name"), "player_id", "left")
    .join(species.select("species_id", "species_name"), "species_id", "left")
    .join(maps.select("map_id", "map_name"), "map_id", "left")
    .join(rods.select("rod_id", "rod_name"), "rod_id", "left")
    .join(baits.select("bait_id", "bait_name"), "bait_id", "left")
)

In [0]:
display(events_silver)

In [0]:
events_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .option("path", f"{silver_path}/events/data") \
    .saveAsTable("game_analytics.silver.events")